In [ ]:

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database("bootcamp")
session.use_schema("atp524")  # Replace with your schema.

In [ ]:
import ast
import html
import json
import re
from pathlib import Path
from IPython.display import display, HTML

workspace_dir = Path(".")
defined_funcs = {}
all_calls = {}
sql_texts = []  # Stores (file_path, sql_content) for cross-referencing

def summarize_body(func_node):
    """Generates a short summary by inspecting calls and SQL strings inside a function."""
    calls = []
    sql_hints = []
    returns_val = False

    for child in ast.walk(func_node):
        if isinstance(child, ast.Return) and child.value is not None:
            returns_val = True

        if isinstance(child, ast.Call):
            if isinstance(child.func, ast.Name):
                calls.append(child.func.id)
            elif isinstance(child.func, ast.Attribute):
                if isinstance(child.func.value, ast.Name):
                    calls.append(f"{child.func.value.id}.{child.func.attr}")
                else:
                    calls.append(child.func.attr)

        if isinstance(child, ast.Constant) and isinstance(child.value, str):
            upper_val = child.value.upper()
            for kw in ["AI_COMPLETE", "CORTEX.COMPLETE", "AI_EMBED", "CORTEX.SEARCH", 
                       "SELECT ", "INSERT ", "MERGE ", "CREATE ", "COPY INTO"]:
                if kw in upper_val and kw.strip() not in sql_hints:
                    sql_hints.append(kw.strip())

    ignore = {"print", "len", "str", "int", "float", "list", "dict", "set", "range", "enumerate", "isinstance"}
    unique_calls = []
    for c in calls:
        base = c.split(".")[-1]
        if base not in ignore and c not in unique_calls:
            unique_calls.append(c)

    parts = []
    if sql_hints:
        parts.append(f"Runs SQL/Cortex ({', '.join(sql_hints[:3])})")
    if unique_calls:
        parts.append(f"Calls {', '.join(unique_calls[:4])}")
    if not parts:
        parts.append("Performs local Python logic/formatting")

    summary = "[Auto] " + "; ".join(parts)
    if returns_val:
        summary += " → returns result"
    return summary

def parse_python_code(code_str, file_path):
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return
    
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            args = [a.arg for a in node.args.args]
            raw_doc = ast.get_docstring(node)
            if raw_doc:
                doc = raw_doc.strip().splitlines()[0]
                is_auto = False
            else:
                doc = summarize_body(node)
                is_auto = True

            defined_funcs[node.name] = {
                "file": str(file_path),
                "line": node.lineno,
                "args": ", ".join(args),
                "doc": doc,
                "is_auto": is_auto
            }
        elif isinstance(node, ast.Call):
            func_name = None
            if isinstance(node.func, ast.Name):
                func_name = node.func.id
            elif isinstance(node.func, ast.Attribute):
                func_name = node.func.attr
            if func_name:
                all_calls.setdefault(func_name, set()).add(str(file_path))

# 1. Collect definitions and calls across .py, .ipynb, and .sql files
for path in workspace_dir.rglob("*"):
    if ".ipynb_checkpoints" in str(path):
        continue
    if path.suffix == ".py":
        parse_python_code(path.read_text(encoding="utf-8", errors="ignore"), path)
    elif path.suffix == ".sql":
        sql_texts.append((str(path), path.read_text(encoding="utf-8", errors="ignore")))
    elif path.suffix == ".ipynb":
        try:
            nb = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
            py_cells = []
            nb_sql_cells = []
            for cell in nb.get("cells", []):
                if cell.get("cell_type") == "code":
                    src = "".join(cell.get("source", []))
                    # Check if cell is a Snowflake Notebook SQL cell
                    lang = cell.get("metadata", {}).get("language", "")
                    if lang == "sql" or src.lstrip().startswith("%%sql"):
                        nb_sql_cells.append(src)
                    else:
                        py_cells.append(src)
            if py_cells:
                parse_python_code("\n".join(py_cells), path)
            if nb_sql_cells:
                sql_texts.append((str(path), "\n".join(nb_sql_cells)))
        except Exception:
            pass

# 2. Match defined function names inside SQL files and SQL notebook cells (case-insensitive)
for func_name in defined_funcs:
    pattern = re.compile(rf"\b{re.escape(func_name)}\s*\(", re.IGNORECASE)
    for file_path, content in sql_texts:
        if pattern.search(content):
            all_calls.setdefault(func_name, set()).add(file_path)

# 3. Build HTML table and Markdown export
rows_html = []
md_lines = [
    f"### Workspace Helper Functions ({len(defined_funcs)} Found)",
    "| Function | Defined In | Purpose / Summary | Used In |",
    "| :--- | :--- | :--- | :--- |"
]

for name, info in sorted(defined_funcs.items()):
    used_list = sorted(all_calls.get(name, []))
    
    sig_html = f"<code>{html.escape(name)}({html.escape(info['args'])})</code>"
    loc_html = f"<code>{html.escape(info['file'])}</code> (L{info['line']})"
    
    escaped_doc = html.escape(info["doc"])
    if info["is_auto"]:
        doc_html = f"<span style='color: #555;'><em>{escaped_doc}</em></span>"
    else:
        doc_html = escaped_doc

    used_html = "<br>".join(f"<code>{html.escape(f)}</code>" for f in used_list) if used_list else "<em>Unused</em>"
    
    rows_html.append(f"""
        <tr>
            <td style="width: 24%; word-break: break-word; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{sig_html}</td>
            <td style="width: 18%; word-break: break-word; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{loc_html}</td>
            <td style="width: 40%; word-break: normal; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{doc_html}</td>
            <td style="width: 18%; word-break: break-word; vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd;">{used_html}</td>
        </tr>
    """)
    
    used_md = ", ".join(f"`{f}`" for f in used_list) or "*Unused*"
    md_lines.append(f"| `{name}({info['args']})` | `{info['file']}` (L{info['line']}) | {info['doc'].replace('|', '\\|')} | {used_md} |")

table_html = f"""
<h3>Workspace Helper Functions ({len(defined_funcs)} Found)</h3>
<table style="width: 100%; table-layout: fixed; border-collapse: collapse; text-align: left; font-size: 13px;">
    <thead>
        <tr style="border-bottom: 2px solid #888;">
            <th style="width: 24%; padding: 8px;">Function</th>
            <th style="width: 18%; padding: 8px;">Defined In</th>
            <th style="width: 40%; padding: 8px;">Purpose / Summary</th>
            <th style="width: 18%; padding: 8px;">Used In</th>
        </tr>
    </thead>
    <tbody>
        {''.join(rows_html)}
    </tbody>
</table>
"""

display(HTML(table_html))
Path("helper_functions_guide.md").write_text("\n".join(md_lines), encoding="utf-8")

## prints the character count written to the markdown file from the write_text() function
